## Uploading Vectors and Vector Index creation in Redis

### Installing Libraries and Utilities

In [ ]:
%pip install python-dotenv redis==8.0.0 openai==2.38.0 numpy

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# loading redis configurations
redis_hostname = os.getenv("REDIS_HOSTNAME")
redis_password = os.getenv("REDIS_PASSWORD")

# loading azure openai configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
embeddings_model_name = os.getenv("EMBEDDINGS_MODEL_NAME")

### Setting up the Redis Client

In [ ]:
import redis

redis_client = redis.Redis(
    host = redis_hostname,
    port = 10000,
    ssl = True,
    decode_responses = True,
    password = redis_password
)

### Create the Vector Index in Redis

In [ ]:
from redis.commands.search.field import (
    TextField,
    TagField,
    NumericField,
    VectorField
)
from redis.commands.search.index_definition import (
    IndexDefinition,
    IndexType
)

# Define the schema for the Vector Index
schema = (
    TextField("id"),

    TextField("title"),
    TextField("content"),

    TagField("category"),
    TagField("difficulty"),
    TagField("service"),
    TagField("author"),

    NumericField("publishedYear"),

    VectorField(
        "embedding",
        "HNSW",
        {
            "TYPE": "FLOAT32",
            "DIM": 1536,
            "DISTANCE_METRIC": "COSINE"
        }
    )
)

# Create the Vector Index finally
redis_client.ft("idx:azure-ai-docs").create_index(
    fields=schema,
    definition=IndexDefinition(
        prefix=["doc:"],
        index_type=IndexType.HASH
    )
)

print("Index created successfully!")

### Create the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_api_key,
    api_version="2024-02-15-preview",
    azure_endpoint=azure_openai_endpoint
)

### Create the Embedding Generator Function

In [ ]:
def generate_embeddings(client, text):
    
    response = client.embeddings.create(
        input=text,
        model = embeddings_model_name
    )
    
    embeddings=response.model_dump()
    return embeddings['data'][0]['embedding']
    

### Populate the JSON Dataset with Vector Embeddings

In [ ]:
import json

with open("./data.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

for item in dataset:
    item["embedding"] = generate_embeddings(
        azure_openai_client,
        item["content"]
    )

with open(
    "./data_with_embeddings.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(dataset, f, indent=2)

print("Embeddings generated successfully.")

### Load Data with Embeddings to Redis as Hash Set

In [ ]:
import json
import numpy as np

with open(
    "./data_with_embeddings.json",
    "r",
    encoding="utf-8"
) as f:

    dataset = json.load(f)

pipe = redis_client.pipeline(transaction=False)

for doc in dataset:

    embedding_bytes = np.array(
        doc["embedding"],
        dtype=np.float32
    ).tobytes()

    pipe.hset(
        f"doc:{doc['id']}",
        mapping={
            "id": doc["id"],
            "title": doc["title"],
            "content": doc["content"],
            "category": doc["category"],
            "difficulty": doc["difficulty"],
            "service": doc["service"],
            "author": doc["author"],
            "publishedYear": doc["publishedYear"],
            "embedding": embedding_bytes
        }
    )

pipe.execute()

print(
    f"Successfully loaded {len(dataset)} documents into Redis."
)

### Verify document exists

In [ ]:
print(redis_client.hget("doc:doc001", "id"))
print(redis_client.hget("doc:doc001", "title"))
print(redis_client.hget("doc:doc001", "category"))